# A population of planets, extended

A reviewed notebook for the Gravitas compute page. It reads one `gravitas.experiment-result/1`, the file the experiment runner exports, handed to it as the text `gravitas_input`, and its three settings as the text `gravitas_params`. The experiment must vary one setting over a seeded uniform draw and observe `distance_to_primary`; the example is twenty planets of the Binary Planet Lab, each started at a drawn distance from Star A, each run by the engine for the same time.

What it does with the twenty trials, in order:

1. **Classify.** A planet is *bound* if its trial finished (`ok`) and its mean distance from the star stayed within `cutoff` times the distance it started at. The start is the setting times a scale the notebook reads from the smallest-setting trials, so no unit is assumed. A trial that was lost or did not finish is not bound; a trial with no usable distance is left out and counted.
2. **Fit.** The chance of being bound as a logistic curve of the setting, `1 / (1 + exp((a - ac) / w))`, by maximum likelihood on a grid of `ac` and `w`.
3. **Resample.** The same fit on 400 bootstrap resamples of the trials, which is where every uncertainty below comes from.
4. **Extend.** Draw `size` synthetic planets whose setting follows a power law, number proportional to `a` to the power minus `slope`, between the smallest and largest setting the experiment drew (slope 0 is the experiment's own uniform sampling; 1 is equal numbers per factor of distance). Give each the fitted chance, and report the expected bound fraction and count.

It returns an `envelope` (six numbers with units and uncertainties), a `table` of ten bins of the setting and a `figure`. Everything random is seeded, so a run is repeatable.

What the extension is: a prediction from twenty points and one assumed curve. Where the experiment drew no planets, the curve is the notebook's assumption, not the engine's answer; the bootstrap interval counts only the scatter of the twenty, not the choice of curve.

In [ ]:
# Read the input and the three settings.
import json, hashlib
import numpy as np

doc = json.loads(gravitas_input)
params = json.loads(gravitas_params)
assert doc["format"] == "gravitas.experiment-result" and doc["formatVersion"] == 1, "this notebook reads gravitas.experiment-result/1"
SLOPE = float(params.get("slope", 1))
SIZE = int(round(float(params.get("size", 200000))))
CUT = float(params.get("cutoff", 3))
assert -1 <= SLOPE <= 3, "slope is between -1 and 3"
assert 10000 <= SIZE <= 500000, "size is between 10000 and 500000"
assert 1.5 <= CUT <= 10, "cutoff is between 1.5 and 10"
input_digest = hashlib.sha256(gravitas_input.encode("utf-8")).hexdigest()

manifest = doc["manifest"]
assert len(manifest["vary"]) == 1 and "distribution" in manifest["vary"][0], "the experiment must vary one setting over a seeded draw"
KEY = manifest["vary"][0]["parameter"]
assert "distance_to_primary" in manifest["observables"]["metrics"], "the experiment must observe distance_to_primary"
LO, HI = float(manifest["vary"][0]["distribution"]["min"]), float(manifest["vary"][0]["distribution"]["max"])
rows = []
left_out = 0
for t in doc["trials"]:
    d = t["results"].get("distance_to_primary")
    if t["status"] in ("ok", "lostBody") and (d is not None or t["status"] == "lostBody"):
        rows.append((float(t["params"][KEY]), t["status"], None if d is None else float(d)))
    else:
        left_out += 1
rows.sort(key=lambda r: r[0])
assert len(rows) >= 10, "fewer than ten usable trials"
a = np.array([r[0] for r in rows])
print(f"{len(rows)} usable trials of {len(doc['trials'])} ({left_out} left out); {KEY} from {a.min():.3f} to {a.max():.3f}")

In [ ]:
# Classify: bound is finished and stayed within CUT times the starting distance.
few = rows[: max(3, len(rows) // 5)]
scale = float(np.median([r[2] / r[0] for r in few if r[1] == "ok" and r[2] is not None]))
y = np.array([1.0 if (r[1] == "ok" and r[2] is not None and r[2] <= CUT * scale * r[0]) else 0.0 for r in rows])
n = len(y)
print(f"start distance = {scale:.4g} x the setting; {int(y.sum())} of {n} bound at cutoff {CUT:g}")
if y.sum() == 0 or y.sum() == n:
    raise ValueError("every planet is bound, or none is: the cutoff leaves nothing to fit")

In [ ]:
# Fit P(bound) = 1 / (1 + exp((a - ac) / w)) on a grid, then the same on 400 resamples.
AC = np.linspace(LO, HI, 301)
W = np.geomspace(0.002, 0.1, 25)
grid_ac, grid_w = np.meshgrid(AC, W, indexing="ij")
z = (a[None, :] - grid_ac.reshape(-1, 1)) / grid_w.reshape(-1, 1)
p = 1.0 / (1.0 + np.exp(np.clip(z, -60, 60)))
p = np.clip(p, 1e-9, 1 - 1e-9)
ll = y[None, :] * np.log(p) + (1 - y[None, :]) * np.log(1 - p)
best = int(np.argmax(ll.sum(axis=1)))
ac_fit, w_fit = float(grid_ac.reshape(-1)[best]), float(grid_w.reshape(-1)[best])
rng = np.random.default_rng(20261011)
BOOT = 400
counts = rng.multinomial(n, np.full(n, 1.0 / n), size=BOOT)
picked = np.argmax(counts @ ll.T, axis=1)
ac_boot, w_boot = grid_ac.reshape(-1)[picked], grid_w.reshape(-1)[picked]
logistic = lambda x, ac, w: 1.0 / (1.0 + np.exp(np.clip((x - ac) / w, -60, 60)))
print(f"a_c = {ac_fit:.4f} +/- {ac_boot.std(ddof=1):.4f}; w = {w_fit:.4f} (bootstrap median {np.median(w_boot):.4f})")

In [ ]:
# Extend: SIZE synthetic planets drawn from a power law in the setting.
u = rng.random(SIZE)
if abs(SLOPE - 1.0) < 1e-9:
    synth = LO * (HI / LO) ** u
else:
    e = 1.0 - SLOPE
    synth = (LO ** e + u * (HI ** e - LO ** e)) ** (1.0 / e)
edges = np.linspace(LO, HI, 201)
hist, _ = np.histogram(synth, edges)
mid = 0.5 * (edges[1:] + edges[:-1])
frac = lambda ac, w: float((hist * logistic(mid, ac, w)).sum() / SIZE)
f_fit = frac(ac_fit, w_fit)
f_boot = np.array([frac(c, v) for c, v in zip(ac_boot, w_boot)])
f_sd = float(f_boot.std(ddof=1))
f_sample = float(y.mean())
f_sample_sd = float(np.sqrt(f_sample * (1 - f_sample) / n))
print(f"bound fraction: the twenty trials {f_sample:.3f}; the {SIZE} synthetic planets {f_fit:.3f} +/- {f_sd:.3f} (slope {SLOPE:g})")

In [ ]:
# What the notebook returns: an envelope, a table and a figure.
LICENSE = {"status": "generated", "statement": "Computed by Gravitas from the file it was given; the notebook is MIT, and the file keeps whatever terms its author gave it.", "basis": "the file is the reader's own or the example Gravitas wrote"}
sigma = lambda s: {"kind": "sigma", "sigma": float(s), "basis": "data"}
envelope = {
    "format": "gravitas.artifact", "formatVersion": 1,
    "id": f"compute:population-at-scale:{input_digest[:16]}",
    "made": {"app": "gravitas"},
    "source": {"kind": "analysis", "id": "compute/population-at-scale", "version": "1", "digest": input_digest},
    "provenance": {
        "credit": "Gravitas experiment result, extended by the reviewed notebook population-at-scale",
        "license": LICENSE,
        "retrieved": "2026-10-11",
        "citations": ["Efron and Tibshirani 1993, An Introduction to the Bootstrap, Chapman and Hall"],
        "reductions": [
            f"{len(rows)} trials; bound = finished and mean distance_to_primary within {CUT:g} x {scale:.4g} x {KEY}.",
            "Logistic curve in the setting by maximum likelihood on a grid (ac, w); 400 bootstrap resamples of the trials, seeded.",
            f"{SIZE} synthetic planets, number proportional to {KEY}^-{SLOPE:g} between {LO:g} and {HI:g}, each given the fitted chance.",
        ],
    },
    "quantities": [
        {"id": "a.crit", "value": ac_fit, "unit": "", "origin": "fitted", "uncertainty": sigma(ac_boot.std(ddof=1))},
        {"id": "a.width", "value": w_fit, "unit": "", "origin": "fitted", "uncertainty": sigma(w_boot.std(ddof=1))},
        {"id": "bound.sample", "value": f_sample, "unit": "", "origin": "measured", "uncertainty": sigma(f_sample_sd)},
        {"id": "bound.extended", "value": f_fit, "unit": "", "origin": "derived", "uncertainty": sigma(f_sd)},
        {"id": "bound.count", "value": SIZE * f_fit, "unit": "count", "origin": "derived", "uncertainty": sigma(SIZE * f_sd)},
        {"id": "trials", "value": float(n), "unit": "count", "origin": "measured", "uncertainty": {"kind": "none"}},
    ],
    "warnings": [
        "the curve is the notebook's assumption: the interval counts the scatter of the trials, not the choice of a logistic",
        "a sample with no planets between its last bound and first unbound trial cannot place the edge better than that gap, however the bootstrap looks",
        "the extension is only as good as the experiment's range: no synthetic planet is outside it",
    ],
}

if w_fit <= W[0] * 1.001:
    envelope["warnings"].append("the fitted width is the smallest the grid allows: the trials are a clean step, so the width says only that the edge is sharper than the grid, not how sharp")

bins = np.linspace(LO, HI, 11)
bin_of = lambda x: np.clip(np.digitize(x, bins) - 1, 0, 9)
tr_n = np.bincount(bin_of(a), minlength=10)
tr_b = np.bincount(bin_of(a), weights=y, minlength=10)
sy_n = np.bincount(bin_of(synth), minlength=10)
centers = 0.5 * (bins[1:] + bins[:-1])
table = {
    "format": "gravitas.observation", "formatVersion": 1, "kind": "table",
    "id": f"compute:population-at-scale:{input_digest[:16]}:bins", "title": f"{len(rows)} trials and {SIZE} synthetic planets in ten bins of {KEY}",
    "object": {"name": manifest.get("title", "experiment")}, "facility": {"name": "Gravitas compute page"},
    "origin": "model",
    "source": {"kind": "import", "id": "gravitas.experiment-result", "version": "1"},
    "credit": envelope["provenance"]["credit"], "license": LICENSE, "retrieved": "2026-10-11",
    "citations": [{"text": envelope["provenance"]["citations"][0]}],
    "reductions": envelope["provenance"]["reductions"],
    "axes": {"x": "setting", "y": "model"},
    "columns": [
        {"id": "setting", "name": KEY, "unit": "", "role": "x", "values": [float(v) for v in centers]},
        {"id": "trials", "name": "trials", "unit": "count", "role": "value", "values": [float(v) for v in tr_n]},
        {"id": "bound", "name": "bound trials", "unit": "count", "role": "value", "values": [float(v) for v in tr_b]},
        {"id": "model", "name": "fitted chance of being bound", "unit": "", "role": "value", "values": [float(v) for v in logistic(centers, ac_fit, w_fit)]},
        {"id": "synthetic", "name": "synthetic planets", "unit": "count", "role": "value", "values": [float(v) for v in sy_n]},
    ],
    "masks": [], "annotations": [],
}

import io, base64
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
xs = np.linspace(LO, HI, 200)
curves = np.array([logistic(xs, c, v) for c, v in zip(ac_boot[:200], w_boot[:200])])
fig, ax = plt.subplots(figsize=(6, 3.5), dpi=100)
ax.fill_between(xs, np.percentile(curves, 5, axis=0), np.percentile(curves, 95, axis=0), color="0.85", label="bootstrap 5 to 95 percent")
ax.plot(xs, logistic(xs, ac_fit, w_fit), color="0.2", label="fitted chance")
ax.plot(a, y, "o", ms=4, color="C0", label="the trials (1 bound, 0 not)")
ax.set_xlabel(KEY)
ax.set_ylabel("chance of being bound")
ax.legend(fontsize=6)
ax.set_title(f"{len(rows)} trials; {SIZE} synthetic planets, slope {SLOPE:g}", fontsize=8)
fig.tight_layout()
png = io.BytesIO()
# No date or version in the file, so the same run gives the same bytes.
fig.savefig(png, format="png", metadata={"Software": None})
plt.close(fig)

gravitas_output = {
    "envelope": envelope,
    "table": table,
    "figure": {"type": "image/png", "base64": base64.b64encode(png.getvalue()).decode("ascii")},
}
print(len(centers), "rows returned")